# DuckDB — Practical Tour

DuckDB is an **in-process OLAP database**. Think SQLite but built for analytics:
columnar storage, vectorised execution, and zero-config setup.

In this sandbox it fills the gap between Spark (distributed) and pandas (in-memory):

```
pandas        → great for row manipulation, small data
DuckDB        → fast SQL analytics on files and DataFrames, no cluster needed
Spark         → distributed processing, large data that doesn't fit locally
```

### Sections
1. In-memory — query Python objects directly
2. Reading files — CSV, JSON, Parquet with glob patterns
3. DuckDB SQL extras — features you don't get in most databases
4. PostgreSQL — attach and query Postgres from DuckDB
5. Persistent file — save a `.duckdb` store across sessions
6. DuckDB ↔ pandas — round-trip patterns

In [ ]:
import duckdb
import pandas as pd
import json, os, pathlib
from faker import Faker
import random

# Shared data directory (mounted from the host)
DATA = pathlib.Path('/home/jupyter/data')
(DATA / 'duck').mkdir(parents=True, exist_ok=True)

print(f'DuckDB version: {duckdb.__version__}')

## Sample data

Generate data once — all sections below reuse it.

In [ ]:
fake = Faker()
Faker.seed(42)
random.seed(42)

CATS    = ['electronics', 'food', 'clothing', 'travel', 'health']
STATUS  = ['completed', 'pending', 'refunded']
COUNTRY = ['US', 'UK', 'MX', 'SG', 'DE', 'BR', 'FR']

users_df = pd.DataFrame([
    {'user_id': i, 'name': fake.name(), 'country': random.choice(COUNTRY),
     'email': fake.email(), 'signup_date': fake.date_between('-2y', 'today')}
    for i in range(1, 101)
])

txns_df = pd.DataFrame([
    {'txn_id': i, 'user_id': random.randint(1, 100),
     'amount': round(random.uniform(5, 800), 2),
     'category': random.choice(CATS), 'status': random.choice(STATUS),
     'created_at': fake.date_time_between('-90d', 'now')}
    for i in range(1, 2001)
])

print(f'Users:        {len(users_df):,}')
print(f'Transactions: {len(txns_df):,}')

---
## 1 · In-memory — query Python objects directly

DuckDB can query a **pandas DataFrame in place** — no copy, no loading.  
Just reference the variable name inside the SQL string.

In [ ]:
# Query a DataFrame as if it were a table — variable name IS the table name
duckdb.sql("""
    SELECT country, COUNT(*) AS users
    FROM users_df
    GROUP BY ALL
    ORDER BY users DESC
""").show()

In [ ]:
# Join two DataFrames with SQL — no merge(), no index juggling
duckdb.sql("""
    SELECT u.country,
           t.category,
           ROUND(SUM(t.amount), 2)  AS revenue,
           COUNT(*)                 AS tx_count
    FROM txns_df  t
    JOIN users_df u USING (user_id)
    WHERE t.status = 'completed'
    GROUP BY ALL
    ORDER BY revenue DESC
    LIMIT 10
""").df()   # .df() → pandas  |  .pl() → polars  |  .show() → print

In [ ]:
# Use a named connection to register views explicitly
con = duckdb.connect()
con.register('users',   users_df)
con.register('txns',    txns_df)

# Now the names are fixed regardless of the Python variable name
con.sql('SELECT COUNT(*) AS n FROM txns').show()

---
## 2 · Reading files — CSV, JSON, Parquet

DuckDB reads files **lazily** — it only scans what the query needs.  
Glob patterns let you query an entire directory as one table.

In [ ]:
# Write sample files to disk so we can demo file queries
users_df.to_csv(DATA / 'duck/users.csv',  index=False)
txns_df.to_parquet(DATA / 'duck/txns.parquet', index=False)

# Write JSON lines for the JSON demo
with open(DATA / 'duck/events.jsonl', 'w') as f:
    for row in txns_df.head(200).to_dict(orient='records'):
        f.write(json.dumps(row, default=str) + '\n')

print('Files written.')

In [ ]:
# read_csv — DuckDB auto-detects types and header
duckdb.sql("SELECT * FROM read_csv('/home/jupyter/data/duck/users.csv') LIMIT 5").df()

In [ ]:
# read_parquet — glob pattern queries multiple files at once
# e.g. after a Spark job writes partitioned output:
#   read_parquet('/home/jupyter/data/output/transactions/**/*.parquet')
duckdb.sql("""
    SELECT category,
           ROUND(AVG(amount), 2) AS avg_amount,
           COUNT(*)              AS n
    FROM read_parquet('/home/jupyter/data/duck/txns.parquet')
    GROUP BY ALL
    ORDER BY avg_amount DESC
""").df()

In [ ]:
# read_json — handles JSON lines automatically
duckdb.sql("""
    SELECT status, COUNT(*) AS n, ROUND(SUM(amount), 2) AS total
    FROM read_json('/home/jupyter/data/duck/events.jsonl')
    GROUP BY ALL
""").show()

In [ ]:
# Create a persistent view over a file — reuse it in the session
con.sql("CREATE OR REPLACE VIEW txns_file AS SELECT * FROM read_parquet('/home/jupyter/data/duck/txns.parquet')")
con.sql("SELECT COUNT(*) AS rows FROM txns_file").show()

---
## 3 · DuckDB SQL extras

DuckDB extends standard SQL with several quality-of-life features.

In [ ]:
# SUMMARIZE — instant descriptive statistics for every column
con.sql("SUMMARIZE txns").df()

In [ ]:
# GROUP BY ALL — group by every non-aggregated column automatically
con.sql("""
    SELECT country, category, status,
           ROUND(SUM(amount), 2) AS revenue
    FROM txns
    JOIN users USING (user_id)
    GROUP BY ALL
    ORDER BY revenue DESC
    LIMIT 8
""").df()

In [ ]:
# SELECT * EXCLUDE   — return everything except named columns
# SELECT * REPLACE   — return everything but override specific columns
con.sql("""
    SELECT
        * EXCLUDE (email),                            -- drop email
        * REPLACE (ROUND(amount, 0)::INT AS amount)   -- round amount in place
    FROM txns
    JOIN users USING (user_id)
    LIMIT 5
""").df()

In [ ]:
# QUALIFY — filter on window function results (no subquery needed)
# "top transaction per user per category"
con.sql("""
    SELECT user_id, category, amount, created_at
    FROM txns
    WHERE status = 'completed'
    QUALIFY ROW_NUMBER() OVER (PARTITION BY user_id, category ORDER BY amount DESC) = 1
    ORDER BY user_id, category
    LIMIT 10
""").df()

In [ ]:
# PIVOT — reshape rows to columns without a CASE WHEN cascade
con.sql("""
    PIVOT (
        SELECT category, status, ROUND(SUM(amount), 2) AS total
        FROM txns
        GROUP BY ALL
    )
    ON status
    USING SUM(total)
    GROUP BY category
    ORDER BY category
""").df()

In [ ]:
# LIST aggregation — collect values into an array
# "what categories does each user buy from?"
con.sql("""
    SELECT user_id,
           LIST(DISTINCT category ORDER BY category) AS categories,
           COUNT(DISTINCT category)                  AS category_count
    FROM txns
    WHERE status = 'completed'
    GROUP BY user_id
    ORDER BY category_count DESC
    LIMIT 5
""").df()

In [ ]:
# Window functions — running total and 7-day moving average
con.sql("""
    WITH daily AS (
        SELECT DATE_TRUNC('day', created_at)::DATE AS day,
               ROUND(SUM(amount), 2)               AS daily_revenue
        FROM txns
        WHERE status = 'completed'
        GROUP BY 1
    )
    SELECT day,
           daily_revenue,
           ROUND(SUM(daily_revenue)  OVER (ORDER BY day), 2)                          AS running_total,
           ROUND(AVG(daily_revenue)  OVER (ORDER BY day ROWS BETWEEN 6 PRECEDING AND CURRENT ROW), 2) AS ma_7d
    FROM daily
    ORDER BY day DESC
    LIMIT 14
""").df()

---
## 4 · PostgreSQL — attach and query directly

DuckDB's `postgres` extension attaches to the running Postgres container.  
You get full read access (and optionally write) over a standard SQL interface.

In [ ]:
pg_con = duckdb.connect()

pg_con.execute("INSTALL postgres; LOAD postgres;")

pg_con.execute("""
    ATTACH 'dbname=sandbox user=spark password=spark123 host=postgres port=5432'
    AS pg (TYPE POSTGRES, READ_ONLY)
""")

# List tables in the attached database
pg_con.sql("SHOW TABLES FROM pg").show()

In [ ]:
# Query Postgres tables with DuckDB SQL — no psycopg2, no ORM
pg_con.sql("""
    SELECT u.country,
           COUNT(DISTINCT t.id)   AS tx_count,
           ROUND(SUM(t.amount), 2) AS revenue
    FROM pg.transactions t
    JOIN pg.users u ON t.user_id = u.id
    WHERE t.status = 'completed'
    GROUP BY ALL
    ORDER BY revenue DESC
""").df()

In [ ]:
# Cross-source query: join a Postgres table with a local pandas DataFrame
# Register the in-memory DataFrame into this connection first
pg_con.register('local_txns', txns_df)

pg_con.sql("""
    SELECT pg.users.country,
           ROUND(SUM(local_txns.amount), 2) AS generated_revenue
    FROM local_txns
    JOIN pg.users ON local_txns.user_id = pg.users.id
    GROUP BY ALL
    ORDER BY generated_revenue DESC
""").df()

In [ ]:
# Export a Postgres query result straight to Parquet
pg_con.sql("""
    COPY (
        SELECT * FROM pg.transactions
    ) TO '/home/jupyter/data/duck/pg_transactions.parquet' (FORMAT PARQUET)
""")

print('Exported Postgres table to Parquet.')

---
## 5 · Persistent file

A `.duckdb` file stored in the mounted `./data/` directory survives container restarts.  
Treat it as a local data warehouse — write Spark results or Postgres snapshots into it.

In [ ]:
db_path = '/home/jupyter/data/duck/warehouse.duckdb'

wh = duckdb.connect(db_path)

# Materialise a table from the Parquet file
wh.execute("""
    CREATE OR REPLACE TABLE transactions AS
    SELECT * FROM read_parquet('/home/jupyter/data/duck/txns.parquet')
""")

wh.execute("""
    CREATE OR REPLACE TABLE users AS
    SELECT * FROM read_csv('/home/jupyter/data/duck/users.csv')
""")

# Create a summary view
wh.execute("""
    CREATE OR REPLACE VIEW daily_revenue AS
    SELECT DATE_TRUNC('day', created_at)::DATE AS day,
           category,
           ROUND(SUM(amount), 2) AS revenue,
           COUNT(*)              AS tx_count
    FROM transactions
    WHERE status = 'completed'
    GROUP BY ALL
""") 

wh.sql("SHOW TABLES").show()
wh.close()
print(f'Saved to {db_path}')

In [ ]:
# Reopen — tables and views are still there
wh = duckdb.connect(db_path, read_only=True)

wh.sql("""
    SELECT * FROM daily_revenue
    ORDER BY day DESC, revenue DESC
    LIMIT 10
""").df()

---
## 6 · DuckDB ↔ pandas — round-trip patterns

The typical pattern: use **pandas** for row-level wrangling, **DuckDB** for SQL aggregations.

In [ ]:
# pandas → DuckDB SQL → pandas
heavy_query = duckdb.sql("""
    WITH ranked AS (
        SELECT *,
               PERCENT_RANK() OVER (PARTITION BY category ORDER BY amount) AS pct_rank
        FROM txns_df
        WHERE status = 'completed'
    )
    SELECT category,
           ROUND(QUANTILE_CONT(amount, 0.50), 2) AS p50,
           ROUND(QUANTILE_CONT(amount, 0.90), 2) AS p90,
           ROUND(QUANTILE_CONT(amount, 0.99), 2) AS p99,
           COUNT(*) AS n
    FROM ranked
    GROUP BY ALL
    ORDER BY p90 DESC
""").df()   # ← back to pandas for plotting / further wrangling

heavy_query

In [ ]:
# Relation API — build queries programmatically (like a query builder)
rel = (
    duckdb.from_df(txns_df)
          .filter("status = 'completed'")
          .aggregate("category, ROUND(SUM(amount),2) AS revenue, COUNT(*) AS n", "category")
          .order("revenue DESC")
)

# Inspect the SQL it generated
print(rel.query('txns_df', 'SELECT * FROM txns_df'))
rel.df()

In [ ]:
# Export query results to multiple formats
result_df = duckdb.sql("SELECT * FROM txns_df WHERE status = 'completed'").df()

# Parquet
duckdb.sql("COPY (SELECT * FROM txns_df) TO '/home/jupyter/data/duck/export.parquet' (FORMAT PARQUET, COMPRESSION ZSTD)")

# CSV
duckdb.sql("COPY (SELECT * FROM txns_df) TO '/home/jupyter/data/duck/export.csv' (HEADER, DELIMITER ',')")

# JSON
duckdb.sql("COPY (SELECT * FROM txns_df LIMIT 10) TO '/home/jupyter/data/duck/export.json' (FORMAT JSON, ARRAY true)")

print('Exports written to /home/jupyter/data/duck/')

---
## Quick-reference cheatsheet

```python
import duckdb

# ── Connections ──────────────────────────────────────────────────
duckdb.sql("...")                        # default in-memory, one-shot
con = duckdb.connect()                   # reusable in-memory connection
con = duckdb.connect('my.duckdb')        # persistent file
con = duckdb.connect('my.duckdb', read_only=True)

# ── Return formats ───────────────────────────────────────────────
con.sql("...").df()                      # pandas DataFrame
con.sql("...").pl()                      # polars DataFrame
con.sql("...").arrow()                   # PyArrow Table
con.sql("...").fetchall()                # list of tuples
con.sql("...").show()                    # print to console

# ── Reading ──────────────────────────────────────────────────────
read_csv('file.csv')
read_csv('*.csv', union_by_name=True)    # multiple files, auto-union
read_parquet('dir/**/*.parquet')         # recursive glob
read_json('events.jsonl')

# ── Writing ──────────────────────────────────────────────────────
COPY (...) TO 'out.parquet' (FORMAT PARQUET, COMPRESSION ZSTD)
COPY (...) TO 'out.csv'     (HEADER, DELIMITER ',')

# ── Python objects ───────────────────────────────────────────────
con.register('name', df)                 # register pandas/arrow as view
duckdb.from_df(df).sql("...")           # relation from DataFrame
duckdb.arrow(arrow_table)               # relation from Arrow

# ── Attach Postgres ──────────────────────────────────────────────
con.execute("INSTALL postgres; LOAD postgres;")
con.execute("ATTACH 'dbname=... host=...' AS pg (TYPE POSTGRES)")
con.sql("SELECT * FROM pg.my_table")

# ── Handy SQL extensions ─────────────────────────────────────────
SUMMARIZE table_name                     -- column stats
GROUP BY ALL                             -- group by non-aggregated cols
SELECT * EXCLUDE (col1, col2)
SELECT * REPLACE (expr AS col)
QUALIFY <window_condition>               -- filter on window result
PIVOT ... ON col USING agg GROUP BY ...
LIST(col ORDER BY ...)                   -- array aggregation
QUANTILE_CONT(col, 0.95)                 -- exact percentile
```